# The Simpsons: A Data Visualization Journey
### Data Visualization — MDS / MEI | First Practical Work

**Authors:** [Author 1 Name] · [Author 2 Name *(if applicable)*]

---

**Dataset:** [The Simpsons Dataset (Kaggle)](https://www.kaggle.com/datasets/prashant111/the-simpsons-dataset/data)  
**Tools:** Python · Vega-Altair · Streamlit · Pandas · NumPy


## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Cleaning](#2-data-cleaning)
   - 2.1 Episodes
   - 2.2 Characters
   - 2.3 Locations
   - 2.4 Script Lines
3. [Colour Palette & Visual Identity](#3-colour-palette--visual-identity)
4. [Q1 — Evolution of IMDb Ratings](#4-q1--evolution-of-imdb-ratings)
5. [Q2 — Evolution of US Viewership](#5-q2--evolution-of-us-viewership)
6. [Q3 — Correlation: Ratings vs Viewership](#6-q3--correlation-ratings-vs-viewership)
7. [Q4 — Viewership by Airing Weekday](#7-q4--viewership-by-airing-weekday)
8. [Q5 — Season Viewer Patterns](#8-q5--season-viewer-patterns)
9. [Q6 (Extra) — Production Complexity Trends](#9-q6-extra--production-complexity-trends)
10. [Final Streamlit Dashboard](#10-final-streamlit-dashboard)


## 1. Setup & Imports

In [ ]:
import re
import os
import warnings

import pandas as pd
import numpy as np
import altair as alt

warnings.filterwarnings("ignore")
os.makedirs("data/outputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/interactive_visualizations", exist_ok=True)

print("Altair version:", alt.__version__)
print("Pandas version:", pd.__version__)


## 2. Data Cleaning

All raw data is loaded from `data/` and cleaned outputs are written to `data/outputs/`.  
Run each sub-section in order to reproduce the clean datasets.


### 2.1 Episodes (`simpsons_episodes.csv`)

**Steps performed:**
1. Drop irrelevant columns (`image_url`, `video_url`)
2. Strip Wikipedia citation artefacts from Season 28 titles (regex `"?[\d+]$`)
3. Parse `original_air_date` to `datetime64`
4. Derive temporal columns: `air_year`, `air_month`, `air_dayofweek_num`, `air_dayofweek`
5. Cast `imdb_votes` and `views` to nullable `Int64`
6. Assert no duplicate `id` or `(season, number_in_season)` pairs
7. Compute 5-episode centred rolling average `imdb_rating_roll5`
8. Merge per-season aggregates back into episode table
9. Drop Season 28 (single episode, outlier values, incomplete data)


In [ ]:
df = pd.read_csv("data/simpsons_episodes.csv")

# 1. Drop irrelevant columns
df.drop(columns=["image_url", "video_url"], inplace=True)

# 2. Fix malformed titles (Season 28 Wikipedia citation artefacts)
n_fixed = 0
def clean_title(t):
    global n_fixed
    cleaned = re.sub(r'"?\[\d+\]$', '', str(t)).strip()
    if cleaned != t:
        n_fixed += 1
    return cleaned

df["title"] = df["title"].apply(clean_title)
print(f"Titles fixed (citation artefacts removed): {n_fixed}")
assert not df["title"].str.contains(r"\[\d+\]", na=False).any(), \
    "Some titles still contain citation artefacts!"

# 3. Parse air dates
df["original_air_date"] = pd.to_datetime(df["original_air_date"], errors="coerce")
print(f"Unparseable dates: {df['original_air_date'].isnull().sum()}")

# 4. Derive temporal columns
df["air_year"]          = df["original_air_date"].dt.year
df["air_month"]         = df["original_air_date"].dt.month
df["air_dayofweek_num"] = df["original_air_date"].dt.dayofweek
day_map = {0:"Monday",1:"Tuesday",2:"Wednesday",3:"Thursday",
           4:"Friday",5:"Saturday",6:"Sunday"}
df["air_dayofweek"] = df["air_dayofweek_num"].map(day_map)

if "original_air_year" in df.columns:
    df.drop(columns=["original_air_year"], inplace=True)

# 5. Missing value summary
print("\nMissing data in episodes:")
for col in ["imdb_rating", "imdb_votes", "us_viewers_in_millions", "views"]:
    if col in df.columns:
        print(f"  {col:<25}: {df[col].isnull().sum()}")

# 6. Type casting
df["imdb_votes"] = df["imdb_votes"].astype("Int64")
df["views"]      = df["views"].astype("Int64")

# 7. Duplicate checks
assert df.duplicated(subset=["id"]).sum() == 0, "Duplicate episode IDs found!"
assert df.duplicated(subset=["season","number_in_season"]).sum() == 0, \
    "Duplicate (season, episode) pairs found!"

df.sort_values("original_air_date", inplace=True)
df.reset_index(drop=True, inplace=True)

# 8. Rolling average
df["imdb_rating_roll5"] = (
    df["imdb_rating"]
    .rolling(window=5, min_periods=3, center=True)
    .mean()
    .round(3)
)

# 9. Season aggregates merged back
season_agg = df.groupby("season").agg(
    season_avg_imdb    = ("imdb_rating",            "mean"),
    season_avg_viewers = ("us_viewers_in_millions",  "mean"),
    season_n_episodes  = ("id",                      "count"),
).round(3).reset_index()
df = df.merge(season_agg, on="season", how="left")

# 10. Drop Season 28
df = df[df["season"] != 28]

print(f"\nEpisodes dataset final shape: {df.shape}")
df.to_csv("data/outputs/simpsons_episodes_clean.csv", index=False)
print("Saved → data/outputs/simpsons_episodes_clean.csv")


### 2.2 Characters (`simpsons_characters.csv`)

**Steps:** normalise column names · Title Case ALL-CAPS names · map gender codes · assert no duplicate IDs.


In [ ]:
chars = pd.read_csv("data/simpsons_characters.csv")
chars.columns = chars.columns.str.strip().str.lower().str.replace(" ", "_")

def fix_char_name(name):
    if isinstance(name, str) and name.isupper():
        return name.title()
    return name

chars["name"] = chars["name"].apply(fix_char_name)
chars["normalized_name"] = chars["normalized_name"].str.strip().str.lower()

gender_map = {"m": "male", "f": "female"}
chars["gender"] = chars["gender"].map(gender_map)

n_no_gender = chars["gender"].isnull().sum()
print(f"Characters with unknown gender: {n_no_gender} ({n_no_gender/len(chars)*100:.1f}%)")
assert chars.duplicated(subset=["id"]).sum() == 0, "Duplicate character IDs found!"

print(f"Characters clean shape: {chars.shape}")
print(chars["gender"].value_counts(dropna=False))
chars.to_csv("data/outputs/simpsons_characters_clean.csv", index=False)
print("Saved → data/outputs/simpsons_characters_clean.csv")


### 2.3 Locations (`simpsons_locations.csv`)

**Steps:** normalise column names · Title Case · deduplicate by `normalized_name` · assert no duplicate IDs.


In [ ]:
locs = pd.read_csv("data/simpsons_locations.csv")
locs.columns = locs.columns.str.strip().str.lower().str.replace(" ", "_")

def fix_loc_name(name):
    if isinstance(name, str) and name.isupper():
        return name.title()
    return name

locs["name"] = locs["name"].apply(fix_loc_name)
locs["normalized_name"] = locs["normalized_name"].str.strip().str.lower()

n_dup_names = locs.duplicated(subset=["normalized_name"]).sum()
print(f"Duplicate location names (normalised): {n_dup_names}")
locs.drop_duplicates(subset=["normalized_name"], keep="first", inplace=True)
assert locs.duplicated(subset=["id"]).sum() == 0, "Duplicate location IDs found!"

locs.reset_index(drop=True, inplace=True)
print(f"Locations clean shape: {locs.shape}")
locs.to_csv("data/outputs/simpsons_locations_clean.csv", index=False)
print("Saved → data/outputs/simpsons_locations_clean.csv")


### 2.4 Script Lines (`simpsons_script_lines.csv`)

**Steps:** drop fully-empty rows · cast integer IDs to `Int64` · normalise `speaking_line` flag · 
strip citation artefacts from text columns · recompute `word_count` · Title Case name columns ·
validate foreign keys · add derived columns (`is_speaking`, `has_dialogue`, `line_length`, `timestamp_in_s`).


In [ ]:
LINES_PATH = "data/simpsons_script_lines.csv"

if not os.path.exists(LINES_PATH):
    print(f"File not found: {LINES_PATH} — skipping script lines cleaning.")
else:
    lines = pd.read_csv(LINES_PATH, low_memory=False)

    # Drop fully-empty rows
    n_before = len(lines)
    lines.dropna(how="all", inplace=True)
    print(f"Dropped {n_before - len(lines)} fully-empty rows")

    # Type casting
    for col in ["id", "episode_id", "number", "character_id", "location_id"]:
        lines[col] = pd.to_numeric(lines[col], errors="coerce").astype("Int64")
    lines["timestamp_in_ms"] = pd.to_numeric(lines["timestamp_in_ms"], errors="coerce")
    lines["word_count"] = pd.to_numeric(lines["word_count"], errors="coerce").astype("Int64")

    # Normalise speaking_line flag
    def to_bool(val):
        if pd.isna(val): return pd.NA
        if isinstance(val, bool): return val
        if isinstance(val, (int, float)): return bool(val)
        s = str(val).strip().lower()
        if s in ("true","1","yes"): return True
        if s in ("false","0","no"): return False
        return pd.NA

    lines["speaking_line"] = lines["speaking_line"].apply(to_bool).astype("boolean")

    # Clean citation artefacts from text columns
    CITATION_RE = re.compile(r'\[\d+\]')
    def clean_text(val):
        if pd.isna(val): return pd.NA
        s = CITATION_RE.sub("", str(val))
        s = re.sub(r'\s+', ' ', s).strip()
        return s if s else pd.NA

    for col in ["raw_text", "spoken_words", "normalized_text"]:
        if col in lines.columns:
            lines[col] = lines[col].apply(clean_text)

    # Recompute word_count
    def count_words(val):
        if pd.isna(val): return pd.NA
        tokens = str(val).split()
        return len(tokens) if tokens else pd.NA

    lines["word_count"] = lines["normalized_text"].apply(count_words).astype("Int64")

    # Title Case name columns
    def fix_name(val):
        if pd.isna(val): return val
        s = str(val).strip()
        return s.title() if s.isupper() else s

    for col in ["raw_character_text", "raw_location_text"]:
        if col in lines.columns:
            lines[col] = lines[col].apply(fix_name)

    # Derived columns
    lines["is_speaking"]  = lines["speaking_line"].astype("boolean")
    lines["has_dialogue"] = lines["spoken_words"].notna()
    lines["line_length"]  = lines["spoken_words"].apply(
        lambda v: len(str(v)) if pd.notna(v) else pd.NA).astype("Int64")
    lines["timestamp_in_s"] = (lines["timestamp_in_ms"] / 1000).round(2)

    lines.sort_values(["episode_id", "number"], inplace=True)
    lines.reset_index(drop=True, inplace=True)

    assert lines.duplicated(subset=["id"]).sum() == 0, "Duplicate script-line IDs!"
    print(f"Final script lines shape: {lines.shape}")
    lines.to_csv("data/outputs/simpsons_script_lines_clean.csv", index=False)
    print("Saved → data/outputs/simpsons_script_lines_clean.csv")


## 3. Colour Palette & Visual Identity

A consistent `HOMER_COLOR_SCHEME` is used across **all** charts and the Streamlit app,
ensuring the same colour always carries the same meaning.
Typically defined in `conf.py` — reproduced here for self-contained execution.


In [ ]:
# Simpsons-inspired, accessibility-checked palette
HOMER_COLOR_SCHEME = {
    "primary":            "#FED90F",   # Simpson Yellow — main trend lines, bars
    "secondary":          "#009DDC",   # Sky Blue       — deviation bands, secondary series
    "tertiary":           "#F77F00",   # Orange         — background dots, scatter
    "alternative_accent": "#D4551A",   # Rust/Red       — annotations, mean labels
}

PRIMARY   = HOMER_COLOR_SCHEME["primary"]
SECONDARY = HOMER_COLOR_SCHEME["secondary"]
TERTIARY  = HOMER_COLOR_SCHEME["tertiary"]
ACCENT    = HOMER_COLOR_SCHEME["alternative_accent"]

def gradient(c1, c2, n):
    """Return n hex colours interpolated linearly between c1 and c2."""
    def hex_to_rgb(h):
        h = h.lstrip("#")
        return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    def rgb_to_hex(r, g, b):
        return f"#{int(r):02X}{int(g):02X}{int(b):02X}"
    r1,g1,b1 = hex_to_rgb(c1)
    r2,g2,b2 = hex_to_rgb(c2)
    if n == 1:
        return [c1]
    return [rgb_to_hex(r1+(r2-r1)*i/(n-1),
                       g1+(g2-g1)*i/(n-1),
                       b1+(b2-b1)*i/(n-1)) for i in range(n)]

SEASON_ORDER  = list(range(1, 28))
SEASON_COLORS = gradient(PRIMARY, TERTIARY, len(SEASON_ORDER))

print("Palette loaded. Colours:")
for k, v in HOMER_COLOR_SCHEME.items():
    print(f"  {k:<22}: {v}")


## 4. Q1 — Evolution of IMDb Ratings

**Question:** How have the IMDb ratings evolved over time?

### Design rationale

Three layers are combined on a single season X axis:
- **Faint episode dots** (opacity 0.33): preserve the full distribution without distracting from the trend
- **1-STD shaded band**: makes within-season variance immediately visible
- **Season average line + data labels**: bold trend with exact values printed above each point so the chart is readable as a static export

Y axis is fixed to `[4, 10]` for direct comparability with Q3.  
The subtitle documents all three layers so no legend is needed.

### Design process
An early version used only the average line, hiding the growing variance in later seasons. 
Adding the STD band revealed that some episodes remain highly-rated even in otherwise 
mediocre seasons. Data labels removed the need for interactive tooltips.


In [ ]:
eps = pd.read_csv("data/outputs/simpsons_episodes_clean.csv")
eps["original_air_date"] = pd.to_datetime(eps["original_air_date"])

season = (
    eps.groupby("season")
    .agg(
        avg_rating  = ("imdb_rating",            "mean"),
        std_rating  = ("imdb_rating",            "std"),
        min_rating  = ("imdb_rating",            "min"),
        max_rating  = ("imdb_rating",            "max"),
        avg_viewers = ("us_viewers_in_millions", "mean"),
        std_viewers = ("us_viewers_in_millions", "std"),
        min_viewers = ("us_viewers_in_millions", "min"),
        max_viewers = ("us_viewers_in_millions", "max"),
        n_episodes  = ("id",                     "count"),
        avg_votes   = ("imdb_votes",             "mean"),
        avg_views   = ("views",                  "mean"),
    )
    .round(3).reset_index()
)
season["avg_rating_lbl"]  = season["avg_rating"].round(1).astype(str)
season["avg_viewers_lbl"] = season["avg_viewers"].round(1).astype(str)

eps_r = eps.dropna(subset=["imdb_rating"]).copy()

TITLE_FONT = 14
AXIS_FONT  = 11

# Layer 1: individual episode dots
dots_r = (
    alt.Chart(eps_r)
    .mark_circle(size=15, opacity=0.33, color=TERTIARY)
    .encode(
        x=alt.X("season:O", title="Season"),
        y=alt.Y("imdb_rating:Q", title="IMDb Rating",
                scale=alt.Scale(domain=[4, 10])),
    )
)

# Layer 2: 1-STD band
band_r = (
    alt.Chart(season.dropna(subset=["avg_rating","std_rating"]))
    .mark_area(opacity=0.15)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("min_r:Q"),
        y2="max_r:Q",
        color=alt.value(SECONDARY),
    )
    .transform_calculate(
        min_r="datum.avg_rating - datum.std_rating",
        max_r="datum.avg_rating + datum.std_rating",
    )
)

# Layer 3: season average line
line_r = (
    alt.Chart(season.dropna(subset=["avg_rating"]))
    .mark_line(strokeWidth=2.5,
               point=alt.OverlayMarkDef(size=50, filled=True))
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("avg_rating:Q"),
        color=alt.value(PRIMARY),
    )
)

# Layer 4: data labels
lbl_r = (
    alt.Chart(season.dropna(subset=["avg_rating"]))
    .mark_text(fontSize=10, fontWeight="bold", dy=-15, color="#444")
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("avg_rating:Q"),
        text=alt.Text("avg_rating:Q", format=".1f"),
    )
)

chart_q1_clean = (
    (dots_r + band_r + line_r + lbl_r)
    .properties(
        width=850, height=400,
        title=alt.TitleParams(
            "Q1 — Evolution of IMDb Ratings by Season",
            subtitle=[
                "Numbers above dots = average season rating.",
                "Line = Season Average · Shaded Area = 1-STD Range",
            ],
            fontSize=16, subtitleFontSize=11, subtitleColor="#444",
        ),
    )
    .configure_view(stroke=None)
    .configure_legend(cornerRadius=5, fillColor="white",
                      strokeColor="#DDD", padding=10)
)

chart_q1_clean.save("outputs/q1_ratings_clean.html")
chart_q1_clean


### Q1 — Alternative / Exploratory Charts

Three additional interactive charts were produced during the exploratory phase.


In [ ]:
# ── VIZ 1: Overview + Zoom brush ─────────────────────────────────────────
brush = alt.selection_interval(encodings=["x"])
base  = alt.Chart(eps)

ov_dots = (
    base.mark_circle(size=25)
    .encode(
        x=alt.X("original_air_date:T", title=None,
                axis=alt.Axis(format="%Y", tickCount="year")),
        y=alt.Y("imdb_rating:Q", title="Rating",
                scale=alt.Scale(domain=[4,10])),
        color=alt.condition(brush, alt.value("#E89B2A"), alt.value("#dddddd")),
        opacity=alt.condition(brush, alt.value(0.75), alt.value(0.3)),
    )
    .add_params(brush)
)
ov_line = (
    base.mark_line(color="#D02020", strokeWidth=1.8, opacity=0.85)
    .encode(x="original_air_date:T",
            y=alt.Y("imdb_rating_roll5:Q", scale=alt.Scale(domain=[4,10])))
)
overview = (ov_dots + ov_line).properties(
    width=820, height=110,
    title="① Drag to select a window — zoom auto-scales below")

zm_dots = (
    base.mark_circle(size=80, opacity=0.80)
    .encode(
        x=alt.X("original_air_date:T", title="Air Date",
                axis=alt.Axis(format="%b %Y", tickCount=10, labelAngle=-35)),
        y=alt.Y("imdb_rating:Q", title="IMDb Rating",
                scale=alt.Scale(zero=False)),
        color=alt.Color("season:O", title="Season",
                        scale=alt.Scale(scheme="tableau20")),
        tooltip=[alt.Tooltip("title:N", title="Episode"),
                 alt.Tooltip("season:O", title="Season"),
                 alt.Tooltip("imdb_rating:Q", title="Rating", format=".1f"),
                 alt.Tooltip("original_air_date:T", title="Air Date", format="%d %b %Y")],
    )
    .transform_filter(brush)
)
zm_line = (
    base.mark_line(strokeWidth=2.2, opacity=0.65, strokeDash=[4,2], color="#D02020")
    .encode(x="original_air_date:T",
            y=alt.Y("imdb_rating_roll5:Q", scale=alt.Scale(zero=False)))
    .transform_filter(brush)
)
zm_avg = (
    base.mark_rule(strokeDash=[6,3], opacity=0.50, color="#555555", strokeWidth=1.5)
    .encode(y=alt.Y("mean(imdb_rating):Q", title=""))
    .transform_filter(brush)
)
zoom = (zm_dots + zm_line + zm_avg).properties(
    width=820, height=310,
    title="② Zoomed — X & Y auto-scale to selection")

viz1 = (
    alt.vconcat(overview, zoom, spacing=14)
    .properties(title="Episode Ratings Over Time – Overview + Zoom (Q1 – Viz 1)")
    .resolve_scale(color="independent")
)
viz1.save("outputs/interactive_visualizations/q1_viz1_scatter_rolling.html")
viz1


In [ ]:
# ── VIZ 2: Click a season bar → episode strip ─────────────────────────────
season_df = (
    eps.groupby("season")
    .agg(avg_rating=("imdb_rating","mean"), n_ep=("id","count"))
    .reset_index().round({"avg_rating":3})
)
season_click = alt.selection_point(fields=["season"])

season_bars = (
    alt.Chart(season_df)
    .mark_bar(cursor="pointer")
    .encode(
        x=alt.X("season:O", title="Season"),
        y=alt.Y("avg_rating:Q", title="Avg IMDb Rating",
                scale=alt.Scale(domain=[5,9])),
        color=alt.condition(
            season_click,
            alt.Color("avg_rating:Q",
                      scale=alt.Scale(scheme="yelloworangered", domain=[5,9]),
                      legend=None),
            alt.value("#dddddd"),
        ),
        tooltip=[alt.Tooltip("season:O"), alt.Tooltip("avg_rating:Q", format=".2f"),
                 alt.Tooltip("n_ep:Q", title="Episodes")],
    )
    .add_params(season_click)
    .properties(width=700, height=260,
                title="Click a season bar to see its episodes ↓")
)

episode_strip = (
    alt.Chart(eps)
    .mark_rect(height=40)
    .encode(
        x=alt.X("number_in_season:O", title="Episode in Season",
                axis=alt.Axis(labelAngle=0)),
        color=alt.Color("imdb_rating:Q", title="IMDb Rating",
                        scale=alt.Scale(scheme="yelloworangered", domain=[4.5,9.5])),
        tooltip=[alt.Tooltip("title:N"), alt.Tooltip("number_in_season:O", title="Ep #"),
                 alt.Tooltip("imdb_rating:Q", title="Rating", format=".1f")],
    )
    .transform_filter(season_click)
    .properties(width=700, height=70, title="Episode ratings for selected season")
)

viz2 = alt.vconcat(season_bars, episode_strip).properties(
    title="Season Avg Ratings – Click to Drill Down (Q1 – Viz 2)")
viz2.save("outputs/interactive_visualizations/q1_viz2_season_bars.html")
viz2


In [ ]:
# ── VIZ 3: Season × Episode heatmap ──────────────────────────────────────
viz3 = (
    alt.Chart(eps, title="Rating Heatmap: Season × Episode Position (Q1 – Viz 3)")
    .mark_rect()
    .encode(
        x=alt.X("number_in_season:O", title="Episode in Season",
                axis=alt.Axis(labelOverlap=True)),
        y=alt.Y("season:O", title="Season", sort="descending"),
        color=alt.Color("imdb_rating:Q", title="IMDb Rating",
                        scale=alt.Scale(scheme="plasma", domain=[4.5,9.5])),
        tooltip=[alt.Tooltip("title:N"), alt.Tooltip("season:O"),
                 alt.Tooltip("number_in_season:O", title="Ep #"),
                 alt.Tooltip("imdb_rating:Q", title="Rating", format=".1f")],
    )
    .properties(width=700, height=450)
)
viz3.save("outputs/interactive_visualizations/q1_viz3_heatmap.html")
viz3


## 5. Q2 — Evolution of US Viewership

**Question:** How have the viewers evolved over time?

### Design rationale
Mirrors Q1's structure for immediate cross-chart comparison:
- **Min-max vertical whiskers** show the full viewership range per season, making outlier episodes visible
- **1-STD shaded band** contextualises within-season variance
- **Season average line + data labels** trace the long-run decline

Y domain fixed to `[0, 35]` millions to include all early-season peaks.

### Design process
An average-only line hid the dramatic range (from ~30M in Season 2 to <5M in Season 27). 
Adding min-max whiskers made the decline visceral. The STD band added within-season context.


In [ ]:
eps_v = eps.dropna(subset=["us_viewers_in_millions"]).copy()
import numpy as np
season["ci_rating"]  = (season["std_rating"]  / np.sqrt(season["n_episodes"]) * 1.96).round(3)

# STD band
band_v = (
    alt.Chart(season.dropna(subset=["avg_viewers","std_viewers"]))
    .mark_area(opacity=0.15)
    .encode(
        x=alt.X("season:O", title="Season",
                axis=alt.Axis(labelAngle=0, titleFontSize=AXIS_FONT)),
        y=alt.Y("min_v:Q", title="US Viewers (millions)",
                scale=alt.Scale(domain=[0,35])),
        y2="max_v:Q",
        color=alt.value(SECONDARY),
    )
    .transform_calculate(
        min_v="datum.avg_viewers - datum.std_viewers",
        max_v="datum.avg_viewers + datum.std_viewers",
    )
)

# Min-max whiskers
range_v = (
    alt.Chart(season.dropna(subset=["min_viewers","max_viewers"]))
    .mark_rule(strokeWidth=1.2, opacity=0.8)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("min_viewers:Q"),
        y2="max_viewers:Q",
        color=alt.value(TERTIARY),
    )
)

# Average line
line_v = (
    alt.Chart(season.dropna(subset=["avg_viewers"]))
    .mark_line(strokeWidth=2.5,
               point=alt.OverlayMarkDef(size=60, filled=True))
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("avg_viewers:Q"),
        color=alt.value(PRIMARY),
    )
)

# Data labels
lbl_v = (
    alt.Chart(season.dropna(subset=["avg_viewers"]))
    .mark_text(fontSize=9, fontWeight="bold", dy=-12, color="#333333")
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("avg_viewers:Q"),
        text=alt.Text("avg_viewers:Q", format=".1f"),
    )
)

chart_q2 = (
    alt.layer(band_v, range_v, line_v, lbl_v)
    .properties(
        width=800, height=400,
        title=alt.TitleParams(
            "Q2 — Evolution of US Viewership by Season",
            subtitle=[
                "Numbers above dots = Avg Viewers (Millions) · Vertical Rules = Min-Max Range",
                "Shaded Area = 1-STD Range",
            ],
            fontSize=TITLE_FONT, subtitleFontSize=10, subtitleColor="#444",
        ),
    )
    .configure_view(stroke=None)
    .configure_legend(strokeColor="#DDD", cornerRadius=5,
                      fillColor="white", padding=10)
)

chart_q2.save("outputs/q2_viewership_final.html")
chart_q2


## 6. Q3 — Correlation: Ratings vs Viewership

**Question:** Is there a correlation between the gradings and the viewers?

### Design rationale
- Each episode is a point: X = viewers, Y = rating, **colour = season** (yellow → blue gradient)
- A NumPy linear regression line is overlaid as a dashed line
- Pearson *r* annotated directly on the chart — no external table needed

The colour encoding reveals that the positive correlation is partly a temporal confound:
early seasons (warm) cluster top-right, late seasons (cool) bottom-left.

### Design process
A neutral single colour showed the trend but hid the temporal driver. 
Colour-encoding by season immediately exposed the confound. 
A `tableau20` scheme was tested but the gradient better communicates continuous time.


In [ ]:
eps_rv = eps.dropna(subset=["imdb_rating","us_viewers_in_millions"]).copy()

x_vals = eps_rv["us_viewers_in_millions"].values
y_vals = eps_rv["imdb_rating"].values
coeffs = np.polyfit(x_vals, y_vals, 1)
x_range = np.linspace(x_vals.min(), x_vals.max(), 80)
reg_df = pd.DataFrame({
    "us_viewers_in_millions": x_range,
    "imdb_rating_fit": coeffs[0]*x_range + coeffs[1],
})
r_value = np.corrcoef(x_vals, y_vals)[0, 1]

def season_color(legend=False):
    return alt.Color(
        "season:O", title="Season",
        scale=alt.Scale(domain=SEASON_ORDER, range=SEASON_COLORS),
        legend=alt.Legend(orient="top") if legend else None,
    )

scatter = (
    alt.Chart(eps_rv)
    .mark_point(size=60, filled=True)
    .encode(
        x=alt.X("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        y=alt.Y("imdb_rating:Q", title="IMDb Rating",
                scale=alt.Scale(domain=[4,10])),
        color=season_color(),
        tooltip=[alt.Tooltip("title:N"), alt.Tooltip("season:O"),
                 alt.Tooltip("imdb_rating:Q", format=".1f"),
                 alt.Tooltip("us_viewers_in_millions:Q", format=".1f", title="Viewers (M)")],
    )
)

reg_line = (
    alt.Chart(reg_df)
    .mark_line(color="#444", strokeWidth=2, strokeDash=[6,4])
    .encode(x="us_viewers_in_millions:Q", y="imdb_rating_fit:Q")
)

reg_annot = (
    alt.Chart(pd.DataFrame([{"x":22,"y":4.2,"label":f"r = {r_value:.2f}"}]))
    .mark_text(fontSize=12, color="#444", fontWeight="bold")
    .encode(x="x:Q", y="y:Q", text="label:N")
)

chart_q3 = (
    (scatter + reg_line + reg_annot)
    .resolve_scale(color="independent")
    .properties(
        width=750, height=400,
        title=alt.TitleParams(
            "Q3 — Correlation: IMDb Rating vs US Viewership",
            subtitle=[
                "Dashed line = linear regression · Warmer colours = earlier seasons.",
                "Bluer points are later seasons; yellower points are from earlier seasons.",
            ],
            fontSize=TITLE_FONT, subtitleFontSize=10, subtitleColor="#444",
        ),
    )
    .configure_legend(fillColor="white", strokeColor="#DDD",
                      cornerRadius=5, padding=10)
    .configure_view(stroke=None)
)

chart_q3.save("outputs/q3_correlation_final.html")
chart_q3


## 7. Q4 — Viewership by Airing Weekday

**Question:** Are the number of viewers for the episodes related to the weekday they were aired?

### Design rationale
The Simpsons aired almost exclusively on **Thursday** (early seasons) and **Sunday** (later seasons).
Other days have too few episodes for a meaningful density estimate and are excluded (noted in subtitle).

- **Violin plots** (density transform, `orient="horizontal"`) reveal the shape of each day's distribution
- **Mean labels** in the accent colour provide a numeric anchor
- X axis (density) has no labels — keeping visual focus on Y position and shape

The violin shapes reveal Thursday is right-skewed (long tail of high-viewership early-season hits)
while Sunday is more concentrated in the 5–15M range.

### Design process
An initial version faceted all seven weekdays — most had <10 episodes, producing noisy curves.
Filtering to Thursday/Sunday improved signal dramatically.
Dot-plot alternatives were tested but violins provided more information per pixel.


In [ ]:
df_ep = pd.read_csv("data/outputs/simpsons_episodes_clean.csv")

WEEKDAY_ORDER  = ["Thursday", "Sunday"]
WEEKDAY_COLOURS = [PRIMARY, TERTIARY]
WEEKDAY_SCALE  = alt.Scale(domain=WEEKDAY_ORDER, range=WEEKDAY_COLOURS)
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
thu_sun   = ["Thursday","Sunday"]

base_q4 = alt.Chart(df_ep).properties(height=260, width=150)

violin = (
    base_q4
    .transform_filter(alt.FieldOneOfPredicate(field="air_dayofweek", oneOf=thu_sun))
    .transform_density(
        "us_viewers_in_millions",
        as_=["us_viewers_in_millions","density"],
        extent=[0, df_ep["us_viewers_in_millions"].max()],
        groupby=["air_dayofweek"],
    )
    .mark_area(orient="horizontal")
    .encode(
        x=alt.X("density:Q").stack("center").impute(None).title(None)
            .axis(labels=False, values=[0], ticks=False, grid=False),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (Millions)"),
        color=alt.Color("air_dayofweek:N", scale=WEEKDAY_SCALE, legend=None),
    )
)

avg_q4 = (
    base_q4
    .transform_aggregate(mean_acc="mean(us_viewers_in_millions)",
                         groupby=["air_dayofweek"])
    .transform_filter(alt.FieldOneOfPredicate(field="air_dayofweek", oneOf=thu_sun))
    .mark_text(fontSize=15, fontWeight="bold", color=ACCENT)
    .encode(
        y=alt.Y("mean_acc:Q"),
        color=alt.value(ACCENT),
        text=alt.Text("mean_acc:Q", format=".1f"),
    )
)

chart_q4 = (
    alt.layer(violin, avg_q4)
    .facet(
        column=alt.Column(
            "air_dayofweek:N", sort=day_order,
            header=alt.Header(title=None, labelOrient="bottom", labelPadding=10),
        )
    )
    .transform_filter(alt.FieldOneOfPredicate(field="air_dayofweek", oneOf=thu_sun))
    .configure_view(stroke=None)
    .properties(
        title=alt.TitleParams(
            "Q4 — Viewership Distribution by Airing Day",
            subtitle=[
                "Violin plot showing distribution of US viewers for Thursday and Sunday airings.",
                "Numbers = average viewership. Other weekdays excluded (insufficient data).",
            ],
            fontSize=TITLE_FONT, subtitleFontSize=10, subtitleColor="#444",
        ),
    )
)

chart_q4.save("outputs/q4_viewers_by_day.html")
chart_q4


## 8. Q5 — Season Viewer Patterns

**Question:** Do the seasons' number of viewers present any relevant pattern?

### Design rationale
Every episode is placed in a grid: **season** on Y, **episode-within-season** on X.  
Cell colour uses the `"blues"` sequential scheme (darker = more viewers).  
Actual values (1 decimal) are printed in each cell, switching to white for very dark cells.

This answers two sub-questions at once:
- **Across seasons**: the gradient from dark (top) to light (bottom) shows the long-run decline
- **Within seasons**: individual cells reveal premiere/finale spikes

### Design process
Initial line charts for mean/max/min per season required a legend and didn't reveal *which* episodes drove extremes. 
The heatmap places all 600+ episodes in a compact view with no overplotting.


In [ ]:
hm_data = (
    eps[["season","number_in_season","us_viewers_in_millions","imdb_rating","title"]]
    .dropna(subset=["us_viewers_in_millions"])
    .copy()
)

heatmap = (
    alt.Chart(hm_data)
    .mark_rect()
    .encode(
        x=alt.X("number_in_season:O", title="Episode Number in Season",
                axis=alt.Axis(labelAngle=0, titleFontSize=AXIS_FONT)),
        y=alt.Y("season:O", title="Season",
                sort=alt.SortOrder("ascending"),
                axis=alt.Axis(titleFontSize=AXIS_FONT)),
        color=alt.Color("us_viewers_in_millions:Q", title="Viewers (M)",
                        scale=alt.Scale(scheme="blues"), legend=None),
        tooltip=[alt.Tooltip("title:N"), alt.Tooltip("season:O"),
                 alt.Tooltip("number_in_season:O", title="Ep #"),
                 alt.Tooltip("us_viewers_in_millions:Q", title="Viewers (M)", format=".1f")],
    )
)

hm_text = (
    alt.Chart(hm_data)
    .mark_text(fontSize=6.5, fontWeight="bold")
    .encode(
        x=alt.X("number_in_season:O"),
        y=alt.Y("season:O", sort=alt.SortOrder("ascending")),
        text=alt.Text("us_viewers_in_millions:Q", format=".1f"),
        color=alt.condition(
            alt.datum.us_viewers_in_millions > 15,
            alt.value("white"),
            alt.value(ACCENT),
        ),
    )
)

chart_q5 = (
    (heatmap + hm_text)
    .properties(
        width=760, height=400,
        title=alt.TitleParams(
            "Q5 — Season Viewer Patterns",
            subtitle=[
                "Cell colour = US viewers (blue, darker = more) · Cell text = viewers in millions",
            ],
            fontSize=TITLE_FONT, subtitleFontSize=9, subtitleColor="#555",
        ),
    )
)

chart_q5.save("outputs/q5_final_heatmap.html")
chart_q5


## 9. Q6 (Extra) — Production Complexity Trends

**Question:** How have production complexity metrics (characters, locations per episode) changed across seasons?

*Requires the script lines dataset.*

### Design rationale
- **Bars**: avg unique **characters** per episode, colour-encoded by avg IMDb rating (gradient scale)
- **Orange line**: avg unique **locations** per episode, normalised to `safe_zone_factor` so it stays in the lower 65% of the chart height. Actual values labelled below each point.

Seasons 27–28 excluded due to missing script data.

### Design process
An early dual-Y-axis version was confusing. Normalising the location line to share the character-count Y axis made the comparison cleaner. 
Colour-encoding bars by rating added a third variable without extra clutter.


In [ ]:
LINES_PATH = "data/outputs/simpsons_script_lines_clean.csv"

if not os.path.exists(LINES_PATH):
    print("Script lines file not found — skipping Q6.")
else:
    lines = pd.read_csv(LINES_PATH, low_memory=False)
    for col in ["episode_id","character_id","location_id","word_count"]:
        lines[col] = pd.to_numeric(lines[col], errors="coerce")

    def to_bool(v):
        if pd.isna(v): return False
        if isinstance(v, bool): return v
        return str(v).strip().lower() in ("true","1","yes")
    lines["speaking_line"] = lines["speaking_line"].apply(to_bool)

    ep_agg = (
        lines.groupby("episode_id")
        .agg(
            n_unique_chars = ("character_id", "nunique"),
            n_unique_locs  = ("location_id",  "nunique"),
        )
        .reset_index()
    )
    eps_q6 = eps.merge(ep_agg.rename(columns={"episode_id":"id"}), on="id", how="left")
    for col in ["n_unique_chars","n_unique_locs"]:
        eps_q6[col] = eps_q6[col].fillna(0).astype(int)

    season_q6 = (
        eps_q6.groupby("season")
        .agg(
            avg_rating = ("imdb_rating",    "mean"),
            avg_chars  = ("n_unique_chars", "mean"),
            avg_locs   = ("n_unique_locs",  "mean"),
        )
        .reset_index()
    )
    season_q6["avg_chars_lbl"] = season_q6["avg_chars"].round(1).astype(str)
    season_q6["avg_locs_lbl"]  = season_q6["avg_locs"].round(1).astype(str)

    season_filtered = season_q6[~season_q6["season"].isin([27,28])].copy()
    max_chars = season_filtered["avg_chars"].max()
    max_locs  = season_filtered["avg_locs"].max()
    safe_zone_factor = (max_chars * 0.65) / max_locs

    color_scale_q6 = alt.Scale(
        domain=[6.5, 8.5],
        range=gradient(TERTIARY, PRIMARY, int((8.5-6.5)*10))
    )

    bar_chars = (
        alt.Chart(season_filtered)
        .mark_bar(opacity=0.4, cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
        .encode(
            x=alt.X("season:O", title="Season",
                    axis=alt.Axis(labelAngle=0, titleFontSize=AXIS_FONT)),
            y=alt.Y("avg_chars:Q",
                    title="Complexity Metrics (Avg per Episode)",
                    scale=alt.Scale(domain=[0, max_chars+5])),
            color=alt.Color("avg_rating:Q", title="Avg IMDb",
                            scale=color_scale_q6,
                            legend=alt.Legend(orient="right")),
        )
    )

    bar_lbl = (
        alt.Chart(season_filtered)
        .mark_text(fontSize=8, dy=-8, fontWeight="bold", color="#333")
        .encode(x=alt.X("season:O"), y=alt.Y("avg_chars:Q"),
                text=alt.Text("avg_chars_lbl:N"))
    )

    line_locs = (
        alt.Chart(season_filtered)
        .mark_line(color=ACCENT, strokeWidth=2.5,
                   point=alt.OverlayMarkDef(size=30, filled=True, color=ACCENT))
        .encode(x=alt.X("season:O"), y=alt.Y("scaled_locs:Q"))
        .transform_calculate(scaled_locs=f"datum.avg_locs * {safe_zone_factor}")
    )

    locs_lbl = (
        alt.Chart(season_filtered)
        .mark_text(fontSize=8, color=ACCENT, fontWeight="bold", dy=12)
        .encode(x=alt.X("season:O"), y=alt.Y("scaled_locs:Q"),
                text=alt.Text("avg_locs_lbl:N"))
        .transform_calculate(scaled_locs=f"datum.avg_locs * {safe_zone_factor}")
    )

    chart_q6 = (
        alt.layer(bar_chars, bar_lbl, line_locs, locs_lbl)
        .properties(
            width=780, height=400,
            title=alt.TitleParams(
                "Q6 — Production Complexity Trends (Seasons 1-26)",
                subtitle=[
                    "Bars = Avg Characters per Episode · Orange Line = Avg Locations per Episode (Normalised)",
                    "Bar colour = Avg IMDb Rating · Seasons 27-28 excluded (missing script data).",
                ],
                fontSize=TITLE_FONT, subtitleFontSize=10, subtitleColor="#444",
            ),
        )
        .configure_view(stroke=None)
    )

    chart_q6.save("outputs/q6_final_complexity.html")
    chart_q6


## 10. Final Streamlit Dashboard

**File:** `streamlit_app.py`

### Layout
All six charts are arranged in a **2-row × 3-column grid** to fit within ~1.5 screen heights:

| Q1 Ratings | Q2 Viewership | Q3 Correlation |
|------------|---------------|----------------|
| Q4 Weekday | Q5 Heatmap    | Q6 Complexity  |

### Cross-chart consistency decisions
- **Single colour palette** (`HOMER_COLOR_SCHEME`) — same colour always carries same meaning
- **Aligned season axis** — Q1, Q2, Q6 all use `season:O` on X; placed in same column for comparison
- **Shared Y scale for ratings** — Q1 and Q3 both fix `[4, 10]`
- **Uniform subtitle style** — `subtitleFontSize=10`, `subtitleColor="#444"` everywhere
- **No redundant legends** — suppressed where colour is self-documenting
- **`configure_view(stroke=None)`** on all charts — removes border box for cleaner embedding

### Running the app


In [ ]:
# Preview the Streamlit app source
with open("streamlit_app.py") as f:
    print(f.read())


```bash
# From the project root:
streamlit run streamlit_app.py
```

All charts are imported directly from the question modules, so any change to a question 
file is automatically reflected in the dashboard on next run.


## Output File Summary

| File | Question | Chart Type |
|------|----------|------------|
| `outputs/q1_ratings_clean.html` | Q1 | Line + band + scatter |
| `outputs/interactive_visualizations/q1_viz1_scatter_rolling.html` | Q1 alt | Overview + zoom brush |
| `outputs/interactive_visualizations/q1_viz2_season_bars.html` | Q1 alt | Bar + episode strip |
| `outputs/interactive_visualizations/q1_viz3_heatmap.html` | Q1 alt | Rating heatmap |
| `outputs/q2_viewership_final.html` | Q2 | Line + band + whiskers |
| `outputs/q3_correlation_final.html` | Q3 | Scatter + regression |
| `outputs/q4_viewers_by_day.html` | Q4 | Faceted violin |
| `outputs/q5_final_heatmap.html` | Q5 | Season viewer heatmap |
| `outputs/q6_final_complexity.html` | Q6 | Bar + normalised line |
